# 18 — Integrated decision-level passing and carrying geometry

**Research objective:** Align already-computed passing and hypothetical carrying geometry at the **same 18,757 possession-start decisions**. Retain the independent 20-match grouping and separate observed coordinates from scenario definitions.

This is an **analytic feature table**, not a model of actual opportunity, suppression, pressing or interceptions. Nothing in these aggregates should be treated as a causal effect or a successful-action prediction.

**Inputs:** `outputs/P2_decision_route_profiles.jsonl` from Notebook 12, and all 20 `outputs/P3_static_carry_corridors_match_*.jsonl` files from Notebook 17. Also read their published gate reports to verify denominators.

**Outputs:** `outputs/P3_integrated_pass_carry_decisions.jsonl` and `outputs/P3_integrated_pass_carry_gate.json`.

Do not rerun the earlier notebooks. A malformed JSONL row, duplicate key, incomplete set of 15 corridors, or any discrepancy in aggregate counts stops execution.


In [ ]:
from pathlib import Path
from collections import Counter,defaultdict
from datetime import datetime,timezone
import json,math,statistics

ROOT=next((p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents)
          if (p/'data'/'matches').is_dir()),None)
if ROOT is None:raise FileNotFoundError('Run from within SkillCorner repository')
OUT=ROOT/'outputs'
PASS=OUT/'P2_decision_route_profiles.jsonl'
FILES=sorted(OUT.glob('P3_static_carry_corridors_match_*.jsonl'))
assert PASS.is_file() and len(FILES)==20,('Missing pass profiles or expected 20 carry files',len(FILES))
pg=json.loads((OUT/'P2_decision_route_profiles_gate.json').read_text())
cg=json.loads((OUT/'P3_static_carry_corridors_gate.json').read_text())
assert pg['totals']['decision_profiles']==18757
assert cg['totals']['corridors']==281355
print('Inputs present:',len(FILES),'carry match files; pass profiles:',PASS.name)


## 1 — Keys and JSONL integrity

The decision identity includes match, period, frame, possession event ID and carrier. No candidate-row independence is assumed. A cross-match duplicate event ID is harmless because match ID is included.

The source carry grid is independently verified against the frozen **15 angle-by-distance combinations**.


In [ ]:
ANGLES=(-60,-30,0,30,60)
LENGTHS=(3.0,5.0,10.0)
GRID={(float(dist),int(angle)) for dist in LENGTHS for angle in ANGLES}
def read_jsonl(path):
    with path.open(encoding='utf-8') as f:
        for line_no,line in enumerate(f,1):
            if not line.strip():raise ValueError(f'Blank JSONL row at {path}:{line_no}')
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f'Malformed JSONL {path}:{line_no}: {e}') from e
def key(r):
    return (str(r['match_id']),int(r['period']),int(r['frame']),
            str(r['possession_event_id']),str(r['carrier_player_id']))
def finite(x):
    if x is None:return None
    try:
        y=float(x)
        return y if math.isfinite(y) else None
    except (ValueError,TypeError):return None
pass_profiles={}
for r in read_jsonl(PASS):
    k=key(r)
    if k in pass_profiles:raise AssertionError(f'Duplicate pass decision: {k}')
    pass_profiles[k]=r
assert len(pass_profiles)==18757


## 2 — Carry decision profiles and strict joins

For each decision, summarize **in-bounds** hypothetical corridors, while also retaining how many of the frozen 15 scenarios exit the pitch. Keep distance measures continuous and allow `null` where there is no detected defender. A detected-defender value does **not** imply that the carrier is detected.

Because distances represent different geometric objects—passing lines to teammates versus hypothetical carry paths—**do not subtract or rank their distance minima as if they shared a calibrated success scale**.


In [ ]:
out_path=OUT/'P3_integrated_pass_carry_decisions.jsonl'
totals=Counter()
per_match=[]
seen=set()
with out_path.open('w',encoding='utf-8') as output:
    for path in FILES:
        mid=path.name.split('_match_')[1].replace('.jsonl','')
        group=defaultdict(list)
        c=Counter()
        for r in read_jsonl(path):
            if str(r['match_id'])!=mid:raise AssertionError(f'{mid}: record match ID mismatch')
            group[key(r)].append(r)
            c['all_corridors']+=1
        for k,rows in group.items():
            if k in seen:raise AssertionError(f'{mid}: duplicate decision across files: {k}')
            seen.add(k)
            profile=pass_profiles.get(k)
            if profile is None:raise AssertionError(f'{mid}: carry decision missing pass profile {k}')
            if len(rows)!=15:raise AssertionError(f'{mid}: expected 15 carry scenarios at {k}, got {len(rows)}')
            got={(float(r['length_m']),int(r['angle_from_attack_axis_deg'])) for r in rows}
            if got!=GRID:raise AssertionError(f'{mid}: missing/duplicate carry scenario at {k}')
            teams={str(r['attacking_team_id']) for r in rows}
            if len(teams)!=1 or next(iter(teams))!=str(profile['attacking_team_id']):
                raise AssertionError(f'{mid}: mismatch in attacking team at {k}')
            for r in rows:
                if finite(r.get('minimum_defender_segment_distance_m')) is None:
                    raise AssertionError(f'{mid}: missing all-defender distance at {k}')
                if not isinstance(r.get('endpoint_within_pitch'),bool):
                    raise AssertionError(f'{mid}: invalid pitch-bound flag at {k}')
            inside=[r for r in rows if r['endpoint_within_pitch']]
            lengths={str(int(d)):{
                'in_pitch_count':sum(float(r['length_m'])==d for r in inside),
                'in_pitch_centerline':any(float(r['length_m'])==d and
                                          int(r['angle_from_attack_axis_deg'])==0 for r in inside)}
                for d in LENGTHS}
            allmin=[finite(r['minimum_defender_segment_distance_m']) for r in inside]
            detmin=[finite(r.get('minimum_detected_defender_segment_distance_m')) for r in inside]
            detmin=[x for x in detmin if x is not None]
            rec={**profile,
                 'carry_scenarios_total':15,
                 'carry_in_pitch_count':len(inside),
                 'carry_out_of_pitch_count':15-len(inside),
                 'carry_in_pitch_by_length_m':lengths,
                 'carry_in_pitch_min_defender_segment_distance_m':min(allmin) if allmin else None,
                 'carry_in_pitch_median_defender_segment_distance_m':statistics.median(allmin) if allmin else None,
                 'carry_in_pitch_detected_defender_distance_defined_count':len(detmin),
                 'carry_in_pitch_min_detected_defender_segment_distance_m':min(detmin) if detmin else None,
                 'carry_forward_endpoint_count':sum(finite(r['signed_endpoint_dx_m'])>0 for r in rows),
                 'feature_interpretation':'joint_static_geometry_only_not_feasible_opportunities'}
            output.write(json.dumps(rec,separators=(',',':'))+'\n')
            c['decision_profiles']+=1
            c['in_pitch_corridors']+=len(inside)
            c['out_of_pitch_corridors']+=15-len(inside)
            c['in_pitch_detected_distance_defined']+=len(detmin)
        per_match.append({'match_id':mid,**dict(c)})
        totals.update(c)
        print(mid,'decisions',c['decision_profiles'],'in bounds',c['in_pitch_corridors'])
assert seen==set(pass_profiles),f'Pass-only decision keys: {len(set(pass_profiles)-seen)}'
assert totals['decision_profiles']==18757,totals
assert totals['all_corridors']==281355,totals
assert totals['out_of_pitch_corridors']==15836,totals
assert totals['in_pitch_corridors']==265519,totals


## 3 — Gate and analytic interpretation

The table is now at the correct **decision grain** for descriptive work, but adjacent decisions may belong to the same attacking possession and match. The next statistical step will need an explicit possession/press-episode framework and match-held-out analysis.

**Scientific constraints:** In-pitch endpoints are merely geometrically valid, not achievable carries. For passing, a teammate being forward does not establish a deliverable pass. A low closest distance is not an interception. Defenders and carriers have not been modelled with acceleration/reaction/control constraints. Outcomes remain separate.


In [ ]:
gate={'created_utc':datetime.now(timezone.utc).isoformat(),
 'purpose':'Join passing and hypothetical carrying geometry at a common possession-start decision unit',
 'n_matches':len(per_match),
 'totals':dict(totals),
 'per_match':per_match,
 'source_gate_counts':{'passing_decisions':pg['totals']['decision_profiles'],
                       'carrying_corridors':cg['totals']['corridors']},
 'constraints':[
  'One record per possession-start decision, NOT independent repeated candidate samples',
  'Fifteen carry scenarios per decision; all scenarios are hypothetical',
  'In-pitch carry endpoints are geometrically valid, NOT feasible carry opportunities',
  'Static pass measures and carry measures cannot be compared as calibrated probabilities',
  'Defender-distance minima are sensitive to the number/quality of observed opponents',
  'Detected-only defender measures do not imply all player positions are detected',
  'Match/possession clustering and future action outcomes must be handled separately',
  'No observed outcomes, pressure classifications, or future frames used to construct features']}
report=OUT/'P3_integrated_pass_carry_gate.json'
report.write_text(json.dumps(gate,indent=2),encoding='utf-8')
print(json.dumps({'matches':gate['n_matches'],'totals':gate['totals']},indent=2))
print('Upload:',report)
